<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/nom_removal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***NOM removal***


In [ ]:
import math as mh
import pandas as pd
import numpy as np

def nom_removal(lanes,Partable2,dailyflow,Treatment):
    # NOM removal  design
    # This code calculates the dimensions of NOM removal stage in a water treatment plant.
    #treatmenttechnologies2 = treatmenttechnologies.astype(str)


    # Input parameters
    print("")
    print("NOM REMOVAL PROCESS")
    flowRate = dailyflow/24  # m3/h

    # Calculate surface loading rate
    if (Treatment['Treatment Technologies'].str.lower().str.strip() == "anionic iex").any():
      nomarea, nomenergy, nomchem,nomhead,nomtank_height=anioniex(flowRate,lanes) #, units
    elif (Treatment['Treatment Technologies'].str.lower().str.strip() == "enhanced coagulation").any():
      nomarea, nomenergy, nomchem,nomhead,nomtank_height=enhchem(flowRate,lanes) #, units

    else:
      nomarea = 0
      nomenergy = 0
      nomchem = 0
      nomhead = 0
      nomtank_height=0

    return nomarea, nomenergy, nomchem, nomhead

def anioniex(flowRate,lanes):
    flowlane=flowRate/lanes # m3/h
    print('We assume 3 vessels per lane, 2 vessels and one regeneratng')
    vessels= 3  # 3 per lane
    working_vessels = 2 # 2 vessels working, 1 regenerating
    flow_m3_hr_per_vessel=flowlane/working_vessels

    # key process parameters
    resin_bulk_density_kg_m3 = 700                  # Resin bulk density [kg/m³]
    bed_porosity = 0.4                              # For head loss calc
    resin_particle_diameter_m = 0.0005              # Typical grain size [m]
    viscosity_water = 1e-3                          # Pa.s
    water_density = 1000                            # kg/m³
    g = 9.81                                        # Gravity [m/s²]
    brine_density_kg_m3 = 1030                      # Approx for 10% NaCl solution
    regen_time_hr = 1                               # 1 hour regeneration time required (assumption)

    # Resin volume required per vessel from EBCT
    while True:
        EBCT_min = int(input("Set the desired Empty Bed Contact Time (EBCT) in minutes (min 3 minutes - max 10minutes, default is 5 minutes): ") or 5)
        if 3 <= EBCT_min <= 10:
            break
        else:
            print("Invalid input. Enter a value between 3 and 10.")
    resin_volume_m3 = flow_m3_hr_per_vessel * (EBCT_min / 60)

    # Trial vessel diameters (we'll choose from a reasonable range)
    possible_diameters = [1.0, 1.2, 1.4, 1.6, 1.8]  # in meters
    best_design = None

    for d in possible_diameters:
        vessel_area_m2 = mh.pi * (d / 2)**2
        bed_height_m = resin_volume_m3 / vessel_area_m2

        if 1.0 <= bed_height_m <= 2.5:
            best_design = {
                'diameter_m': d,
                'area_m2': vessel_area_m2,
                'bed_height_m': bed_height_m,
                'resin_volume_m3': resin_volume_m3
            }
            break  # take the first acceptable one

    # Check if design found
    if best_design is None:
        raise ValueError("No suitable vessel diameter found for target EBCT and practical bed height.")

    # Total resin volume in operation (2 vessels)
    total_resin_m3 = resin_volume_m3 * working_vessels * lanes

    # Total resin area in operation (3 vessels in total + 1.1 factor for safety)
    total_resin_m2 = vessel_area_m2 * vessels * lanes

    # NaCl calculation
    salt_dose_g_per_L = 200  # NaCl regeneration dose [g/L resin]
    NaCl_per_regen_kg_vessel = salt_dose_g_per_L * best_design['resin_volume_m3']/1000  # to kg
    regen_per_day= 1
    nomchem = NaCl_per_regen_kg_vessel * regen_per_day * lanes

    # Brine volume required = salt mass / concentration
    brine_concentration_g_L = 100
    brine_volume_m3 = nomchem / brine_concentration_g_L

    # Brine tank size (with 20% margin)
    tank_volume_m3_day = brine_volume_m3 * 1.2

    # Set the storage time
    while True:
        ndays = int(input("Set the brine storage time in days (min 1 day - max 40days, default is 30days): ") or 30)
        if 1 <= ndays <= 40:
            break
        else:
            print("Invalid input. Enter a value between 1 and 40.")
    # Tank area
    tank_volume_m3_ndays = tank_volume_m3_day * ndays
    tank_height=int(input("Set tank height (range 1-3m)"))
    tank_area_m2=tank_volume_m3_ndays/tank_height

    # Τotal required area
    nomarea= tank_area_m2 + total_resin_m2

    # Pump flow rate
    pump_flow_m3_h = tank_volume_m3_day / regen_time_hr

    # Total Dynamic Head (TDH)
    TDH_m = 2 + 1

    # Brine Pump power [kW]
    pump_power_kW = (brine_density_kg_m3 * g * pump_flow_m3_h * TDH_m/ 3600) / 0.7 # 70% efficiency of the pump

    # Motor power
    motor_power_kW = pump_power_kW / 0.9 # 90% efficiency of the motor
    nomenergy=motor_power_kW *lanes *1

    # Flow velocity and head loss
    flow_m3_s = flow_m3_hr_per_vessel / 3600
    velocity_m_s = flow_m3_s / best_design['area_m2']

    # Reynolds number
    Re = (water_density * velocity_m_s * resin_particle_diameter_m) / viscosity_water

    # Friction factor
    term1 = 150 * ((1 - bed_porosity)**2) / (Re * bed_porosity**3)
    term2 = 1.75 * ((1 - bed_porosity)) / (bed_porosity**bed_porosity)
    friction_factor = term1 + term2

    # Head loss
    head_loss_m = friction_factor * (best_design['bed_height_m'] / resin_particle_diameter_m) * (velocity_m_s**2 / (2 * g))
    nomhead = head_loss_m + 2  # Add safety margin
    nomtank_height=tank_height
    # === RESULTS === #
    print("=== Optimized Anion IEX Design ===")
    print(f"Total plant flow: {dailyflow} m³/day")
    print(f"Operating vessels: {working_vessels} (out of 3)")
    print(f"Flow per vessel: {flow_m3_hr_per_vessel:.2f} m³/h")
    print(f"Target EBCT: {EBCT_min} minutes")
    print(f"Selected vessel diameter: {best_design['diameter_m']} m")
    print(f"Bed height: {best_design['bed_height_m']:.2f} m")
    print(f"Resin volume per vessel: {best_design['resin_volume_m3']:.2f} m³")
    print(f"NaCl required per day: {nomchem:.1f} kg/day")
    print(f"The required daily energy for regeneration is: {nomenergy:.3f} kWh")
    print(f"The required area for the Anion IEX: {nomarea:.1f} m2")
    print(f"Velocity through bed: {velocity_m_s:.4f} m/s")
    print(f"Pump head required (with margin): {nomhead:.2f} m")

    return nomarea, nomenergy, nomchem, nomhead,nomtank_height


def enhchem(flowRate,lanes):

    print("\n--- Chemicals storage area design ---")
    print("Select coagulant:")
    print("1. Aluminium coagulant")
    print("2. Iron coagulant")
    while True:
        coagchem = input("Enter your choice (1/2): ")
        coagchemint=int(coagchem)
        if coagchemint==1 or coagchemint==2:
           break
        else:
           print("Invalid input. Please enter a valid number (1 or 2).")
    if coagchem=="1":
    while True:
        coagdosing=int(input("Set the optimum dosing of alum coagulant (mg/l minimum 30 mg/l):")) # dosing of coaggulant
        if  tank_height >= 30:
            break
        else:
            print("Invalid input. Enter a value higher than 30.")
       coagdensity=2670 #2670 kg/m3 found in the litterature
       coagtype="Alum"
    elif coagchem== "2":
    while True:
        coagdosing=int(input("Set the optimum dosing of iron coagulant (mg/l minimum 35 mg/l):")) # dosing of coaggulant
        if  tank_height >= 35:
            break
        else:
            print("Invalid input. Enter a value higher than 30.")
       coagdensity=2840 #2840 kg/m3 found in the litterature
       coagtype="Iron"
#    CoagTankwidth=int(input("Set the width of the coagulant tank (m):")) # dosing of coaggulant
#    CoagTankunit=CoagTankwidth**2
#    Coag_area=mh.ceil(units*lanes*CoagTankunit)
    amountofcoagday=round(lanes*coagdosing*1000*flowlane/1000000,2) #kg/day
    print("The total amount of coagulant required per day is {:.2f} kg".format(amountofcoagday))
    volumeofcoag=amountofcoagday/coagdensity
    while True:
        ndays = int(input("Set the chemicals storage time in days (min 1day and default is 30days): ") or "30")
        if 1 <= ndays <= 365:
            break
        else:
            print("Invalid input. Enter a value between 1 and 365.")
    totalvolumeofcoag=volumeofcoag*ndays #m3/month
    totalvolumeofcoag=mh.ceil(totalvolumeofcoag)
    nomarea=totalvolumeofcoag/2.5 #assuming 2.5m height
    nomarea=mh.ceil(chemarearequired)
    Pchem = 0.5 # Kw
    nomenergy = 24*Pchem #KWh per day
    nomhead = 1 + (0.001 * nomarea) # head losses in meters
    print("The total volume of coagulant for the storing period is {:.2f} m3".format(totalvolumeofcoag))
    print("The total storage area required is {:.2f} m2".format(nomarea))
    nomtank_height=0
    return nomarea, nomenergy, nomchem,nomhead,nomtank_height